# Chapter 2 — Knowledge Graph Qualification

## 학습 목표
1. GDS 4지표 (Node Similarity / Degree Centrality / Triangles & Clustering / Link Prediction)를 **그래프 품질 진단** 관점으로 해석한다.
2. GDS projection → algorithm → write-back 한 사이클을 DozerDB에서 실행한다.
3. 4개 지표를 `@function_tool`로 노출하고, 4-provider agent가 도구를 어떻게 선택·조합하는지 Opik trace로 비교한다.

## 전제 조건
- Chapter 1 인덱싱 완료 (FinDER 일부 + community_id write-back)
- DozerDB `gds.*` 권한

In [ ]:
from pathlib import Path
import os, sys, json
from dotenv import load_dotenv

HERE = Path.cwd()
if HERE.name != 'teaching-resource' and (HERE / 'teaching-resource').exists():
    os.chdir(HERE / 'teaching-resource')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
for cand in [Path.cwd() / '.env', Path.cwd().parent / '.env']:
    if cand.exists():
        load_dotenv(cand, override=False); break

from _shared.opik_setup import setup_opik, opik_console_link, teardown_opik
from _shared.providers import compare_providers, available_providers, chat, PROVIDERS

trace_info = setup_opik('02', only_jsonl=not os.getenv('OPIK_API_KEY'))
print('Opik project:', trace_info['project'])

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PWD = os.getenv('NEO4J_PASSWORD')
USE_NEO4J = bool(NEO4J_PWD)
DATABASE = os.getenv('CH01_DATABASE') or 'neo4j'  # Ch 1에서 만든 DB 이름과 맞춰주세요
print('database:', DATABASE)

## 2.1 GDS 4지표 — 그래프 품질 진단 매핑

| 지표 | 무엇을 보는가 | 품질 신호 |
|---|---|---|
| **Node Similarity (Jaccard)** | 이웃 집합 겹침 | 중복 엔터티 후보 → 디듀프 트리거 |
| **Degree Centrality** | 연결 수 | hub 노드 식별. 핵심 개체 degree↓ → 추출 누락 의심 |
| **Triangles & Clustering Coef.** | 삼각형 closure | 도메인 응집성. FIBO 같은 분류체계는 높아야 함 |
| **Link Prediction (Adamic-Adar)** | 누락 가능 엣지 | 추출 누락 관계 후보 → 재추출 큐 |

각 지표는 "수치가 이상하면 인덱싱 어느 단계가 깨진 것"인지 진단 신호로 매핑된다.

## 2.2 GDS Pipeline — projection → algorithm → write-back

`elementId()`만 사용 (deprecated `id()` 금지 — CLAUDE.md §8). 모든 in-memory projection은 사용 후 반드시 drop.

In [ ]:
GRAPH = 'ch02-quality'

def gds_run(cypher: str, **params):
    """Ch 2 전용 헬퍼 — neo4j 드라이버로 GDS Cypher 실행."""
    from neo4j import GraphDatabase
    drv = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PWD))
    try:
        with drv.session(database=DATABASE) as s:
            return s.run(cypher, **params).data()
    finally:
        drv.close()

if USE_NEO4J:
    # 기존 projection 정리
    try:
        gds_run(f"CALL gds.graph.drop('{GRAPH}', false)")
    except Exception:
        pass
    # Entity-MENTIONS-Entity 공기 그래프 투영
    gds_run(f'''
      CALL gds.graph.project.cypher(
        '{GRAPH}',
        'MATCH (e) WHERE NOT e:Source AND NOT e:Chunk RETURN id(e) AS id',
        'MATCH (e1)<-[:MENTIONS]-(c:Chunk)-[:MENTIONS]->(e2) WHERE id(e1) < id(e2) RETURN id(e1) AS source, id(e2) AS target'
      )
    ''')
    info = gds_run(f"CALL gds.graph.list('{GRAPH}') YIELD nodeCount, relationshipCount RETURN nodeCount, relationshipCount")
    print('projection:', info)
else:
    print('(neo4j 필요)')

### (a) Node Similarity — 중복 엔터티 후보

In [ ]:
if USE_NEO4J:
    sim = gds_run(f'''
      CALL gds.nodeSimilarity.stream('{GRAPH}')
      YIELD node1, node2, similarity
      WHERE similarity > 0.5
      RETURN gds.util.asNode(node1).name AS a,
             gds.util.asNode(node2).name AS b,
             similarity
      ORDER BY similarity DESC LIMIT 10
    ''')
    for r in sim: print(r)

### (b) Degree Centrality — hub 엔터티

In [ ]:
if USE_NEO4J:
    deg = gds_run(f'''
      CALL gds.degree.stream('{GRAPH}')
      YIELD nodeId, score
      RETURN gds.util.asNode(nodeId).name AS name, score
      ORDER BY score DESC LIMIT 10
    ''')
    for r in deg: print(r)

### (c) Triangles & Clustering Coefficient — 응집성

In [ ]:
if USE_NEO4J:
    clu = gds_run(f'''
      CALL gds.localClusteringCoefficient.stream('{GRAPH}')
      YIELD nodeId, localClusteringCoefficient
      RETURN gds.util.asNode(nodeId).name AS name, localClusteringCoefficient AS clustering
      ORDER BY clustering DESC LIMIT 10
    ''')
    for r in clu: print(r)

### (d) Link Prediction (Adamic-Adar) — 누락 가능 엣지

In [ ]:
if USE_NEO4J:
    pred = gds_run(f'''
      MATCH (a), (b)
      WHERE a <> b AND NOT (a)-[:MENTIONS|HAS_CHUNK|RELATED_TO]-(b)
        AND NOT a:Source AND NOT a:Chunk AND NOT b:Source AND NOT b:Chunk
      WITH a, b LIMIT 200
      RETURN a.name AS a, b.name AS b,
             gds.alpha.linkprediction.adamicAdar(a, b) AS aa
      ORDER BY aa DESC LIMIT 10
    ''')
    for r in pred: print(r)

**이 단계의 진단 체크리스트**
- degree 분포가 power-law를 따르지 않는다 → 추출 단계의 hub 누락
- node similarity > 0.9 페어가 많다 → 자동 머지 후보 (그러나 휴먼 확인 권장)
- clustering coefficient 평균이 < 0.1 → 도메인 응집성 부족, 관계 추출 부실
- 상위 Adamic-Adar 페어가 의미적으로 연결되어 있어야 한다 → 그렇지 않다면 false positive

### 📎 GDS 엔지니어링 깊이 — `chapter-02-gds-engineering.md`

지금까지는 *어떤* 알고리즘을 *왜* 쓰는지에 집중했다. 그래프가 커지면 다음 결정들이 운영 비용을 좌우한다:
- 알고리즘별 시간복잡도 + Java heap 점유 추정
- projection 메모리 추정 공식 + `gds.graph.project.estimate` 사용
- **증분 vs 풀 재계산** 정책 (지표별 트리거 기준)
- `GDSRunMeta` 노드로 "다음 재계산이 언제 필요한가" 추적
- DozerDB 호환성 메모

부속 문서: [`chapter-02-gds-engineering.md`](./chapter-02-gds-engineering.md)

**원칙**: *GDS 결과는 그래프의 진실이 아니라 그래프의 "지금 시점 추정"이다. cache하고, 변화량 트리거로만 갱신한다.*

In [ ]:
# 부속 문서 §2: projection 메모리 사전 추정 — 실제 실행 전 heap 점유 검증
if USE_NEO4J:
    estimate = gds_run('''
      CALL gds.graph.project.estimate(
        'MATCH (e) WHERE NOT e:Source AND NOT e:Chunk RETURN id(e) AS id',
        'MATCH (e1)<-[:MENTIONS]-(c:Chunk)-[:MENTIONS]->(e2)
         WHERE id(e1)<id(e2) RETURN id(e1) AS source, id(e2) AS target',
        {readConcurrency: 4}
      )
      YIELD bytesMin, bytesMax, nodeCount, relationshipCount, requiredMemory
      RETURN bytesMin, bytesMax, nodeCount, relationshipCount, requiredMemory
    ''')
    for r in estimate:
        print(f"projection plan: nodes={r['nodeCount']}, rels={r['relationshipCount']}")
        print(f"  memory: min={r['bytesMin']/1024:.1f}KB  max={r['bytesMax']/1024:.1f}KB  required={r['requiredMemory']}")
else:
    print('(neo4j 필요)')

## 2.3 @function_tool 4종 — agent에게 도구로 노출

각 도구의 docstring이 agent의 tool-selection 정확도를 결정. "언제 호출해야 하는지"를 명확히 적자.

In [ ]:
from agents import function_tool  # OpenAI Agents SDK
from typing import List, Dict, Any

@function_tool
def compute_node_similarity(top_k: int = 10) -> List[Dict[str, Any]]:
    """인덱싱된 엔터티 그래프에서 이름은 다르지만 이웃 집합이 비슷한 페어를 찾는다.
    
    USE WHEN: 사용자가 "중복 엔터티", "머지 후보", "디듀프"를 언급할 때.
    Returns: similarity 내림차순 top-K {a, b, similarity}.
    """
    return gds_run(f'''
      CALL gds.nodeSimilarity.stream('{GRAPH}') YIELD node1, node2, similarity
      WHERE similarity > 0.5
      RETURN gds.util.asNode(node1).name AS a, gds.util.asNode(node2).name AS b, similarity
      ORDER BY similarity DESC LIMIT $k
    ''', k=top_k)

@function_tool
def find_hub_entities(top_k: int = 10) -> List[Dict[str, Any]]:
    """가장 많이 연결된 엔터티 (degree 기준) 를 반환한다.
    
    USE WHEN: "중요 엔터티", "핵심 개체", "추출 누락" 진단할 때.
    """
    return gds_run(f'''
      CALL gds.degree.stream('{GRAPH}') YIELD nodeId, score
      RETURN gds.util.asNode(nodeId).name AS name, score
      ORDER BY score DESC LIMIT $k
    ''', k=top_k)

@function_tool
def graph_clustering_summary() -> Dict[str, Any]:
    """전체 그래프의 응집성 요약 (mean clustering coefficient).
    
    USE WHEN: 그래프 "전반적 품질", "응집성", "도메인 일관성" 판단할 때.
    """
    rows = gds_run(f'''
      CALL gds.localClusteringCoefficient.stream('{GRAPH}')
      YIELD localClusteringCoefficient AS c
      RETURN avg(c) AS mean_clustering, percentileCont(c, 0.5) AS median, count(*) AS n
    ''')
    return rows[0] if rows else {}

@function_tool
def suggest_missing_links(top_k: int = 10) -> List[Dict[str, Any]]:
    """Adamic-Adar 점수가 높지만 아직 연결되지 않은 엔터티 페어를 제안.
    
    USE WHEN: "누락된 관계", "추출 보강", "link prediction" 요구할 때.
    """
    return gds_run('''
      MATCH (a), (b)
      WHERE a <> b AND NOT (a)-[:MENTIONS|RELATED_TO]-(b)
        AND NOT a:Source AND NOT a:Chunk AND NOT b:Source AND NOT b:Chunk
      WITH a, b LIMIT 200
      RETURN a.name AS a, b.name AS b, gds.alpha.linkprediction.adamicAdar(a, b) AS aa
      ORDER BY aa DESC LIMIT $k
    ''', k=top_k)

tools = [compute_node_similarity, find_hub_entities, graph_clustering_summary, suggest_missing_links]
print('tools registered:', [t.name for t in tools])

## 2.4 4-provider Agent — 같은 시나리오, 다른 tool 선택

사용자 시나리오: **"이 그래프 품질을 평가해줘"**

Provider별로 같은 도구를 어떤 순서/조합으로 호출하는지 비교 → Opik trace에서 tool_use 페어가 그대로 보인다.

In [ ]:
from agents import Agent, Runner
from seocho.store.llm import create_llm_backend

SYSTEM = (
    'You are a knowledge graph quality auditor. '
    'Tools available: 4 GDS-backed inspectors. '
    'Call them as needed (parallel when independent), then synthesize a 5-line health report in Korean.'
)
USER = '현재 인덱싱된 FinDER 그래프의 품질을 4가지 관점에서 평가하고 가장 시급한 보완 1가지를 알려줘.'

async def run_agent(provider: str):
    backend = create_llm_backend(provider=PROVIDERS[provider]['provider'], model=PROVIDERS[provider]['model'])
    agent = Agent(
        name=f'quality-auditor-{provider}',
        instructions=SYSTEM,
        tools=tools,
        model=backend.to_agents_sdk_model(),
    )
    result = await Runner.run(agent, USER)
    return result

LIVE_AGENT = os.getenv('CH02_AGENT_RUN', '0') == '1' and USE_NEO4J
if not LIVE_AGENT:
    print('CH02_AGENT_RUN=1 로 환경변수 세팅 후 재실행하면 4-provider agent 비교 수행.')
else:
    import asyncio
    out = {}
    for prov in [n for n, ok in available_providers().items() if ok]:
        try:
            res = asyncio.run(run_agent(prov))
            out[prov] = {
                'final': str(res.final_output)[:400],
                'tool_calls': len([item for item in res.new_items if item.type == 'tool_call_item']),
            }
        except Exception as exc:
            out[prov] = {'error': f'{type(exc).__name__}: {exc}'}
    for prov, payload in out.items():
        print('---', prov, '---')
        print(json.dumps(payload, ensure_ascii=False, indent=2))

**Opik에서 비교할 것**
1. **호출 수**: agent가 4개 도구를 모두 부르는가, 일부만 부르는가? (지나친 over-calling은 prompt가 명확하지 않다는 신호)
2. **병렬화**: 독립적 도구 호출을 직렬화하지 않았는가?
3. **인용**: 도구 결과를 final answer에 실제로 인용했는가? (호출만 하고 결과를 버리는 패턴은 안티)
4. **reasoning depth**: provider별 reasoning step 개수 vs 응답 품질

## 2.5 Cleanup — projection drop

In [ ]:
if USE_NEO4J:
    try:
        gds_run(f"CALL gds.graph.drop('{GRAPH}')")
        print('projection dropped')
    except Exception as exc:
        print('drop skipped:', exc)

teardown_opik()
print('chapter 02 done →', trace_info.get('jsonl_path'))

## 다음 챕터 (Ch 3 — Text2Cypher)
Ch 2 도구는 "수치 진단"에 강하지만 사용자 자연어 질의에는 약하다. Ch 3에서 LLM이 ontology 발췌를 보고 Cypher를 직접 생성하도록 만든 후 4-provider 정확도를 비교한다.